In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.stats import spearmanr

# ----------------------------
# Config
ALPHAS = (0.05, 0.01)  # 95% and 99%
N_SIMS = 100_000  # adjust if needed
RANDOM_SEED = 42
DATA_DIR = Path.cwd() / "testfiles_" / "data"
RETURNS_PATH = DATA_DIR / "test9_1_returns.csv"
PORTFOLIO_PATH = DATA_DIR / "test9_1_portfolio.csv"
IS_PRICE_INPUT = False  # set True if returns file contains PRICES (will convert to log returns)
# ----------------------------

rng = np.random.default_rng(RANDOM_SEED)


def fit_distribution(data: np.ndarray, dist_type: str):
    dist_type = str(dist_type).strip().lower()
    if dist_type in ("normal", "gaussian"):
        mu = np.mean(data)
        sigma = np.std(data, ddof=1)
        return stats.norm(loc=mu, scale=sigma)
    elif dist_type in ("t", "student", "student-t", "student_t"):
        df_t, mu, sigma = stats.t.fit(data)
        return stats.t(df=df_t, loc=mu, scale=sigma)
    else:
        raise ValueError(f"Unknown distribution type: {dist_type}")


def nearest_psd(S: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Eigenvalue clipping to ensure PSD, then re-symmetrize."""
    S = 0.5 * (S + S.T)
    d, V = np.linalg.eigh(S)
    d = np.maximum(d, eps)
    S_psd = (V * d).dot(V.T)
    return 0.5 * (S_psd + S_psd.T)


def quantile_linear(x: np.ndarray, q: float) -> float:
    """np.quantile with fallback for older numpy without 'method' arg."""
    try:
        return float(np.quantile(x, q, method="linear"))
    except TypeError:
        return float(np.quantile(x, q))


def calculate_var_es_from_pnl(pnl: np.ndarray, alpha: float):
    """Return positive VaR/ES magnitudes for loss (left tail)."""
    pnl_sorted = np.sort(pnl)  # ascending (losses negative)
    idx = int(np.floor(alpha * len(pnl_sorted)))
    idx = max(1, min(idx, len(pnl_sorted) - 1))
    var_abs = -pnl_sorted[idx]
    es_abs = -np.mean(pnl_sorted[:idx])
    return var_abs, es_abs


def calculate_var_es_copula(returns_df: pd.DataFrame,
                            portfolio_df: pd.DataFrame,
                            alphas=ALPHAS,
                            n_simulations=N_SIMS,
                            random_seed=RANDOM_SEED,
                            is_price_input: bool = IS_PRICE_INPUT) -> pd.DataFrame:
    rng_local = np.random.default_rng(random_seed)

    # Returns preprocessing
    df = returns_df.copy()
    # Try to ignore a 'Date' column if present
    value_cols = [c for c in df.columns if c.lower() != "date"]
    df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=value_cols, how="all").reset_index(drop=True)

    # Portfolio preprocessing
    pf = portfolio_df.copy()
    required_cols = {"Stock", "Holding", "Starting Price", "Distribution"}
    missing = required_cols - set(pf.columns)
    if missing:
        raise ValueError(f"Portfolio CSV missing columns: {missing}")

    # Ensure returns have all stocks listed in portfolio
    stocks = [s for s in pf["Stock"].tolist() if s in df.columns]
    if len(stocks) == 0:
        raise ValueError("No overlap between portfolio 'Stock' and returns columns.")
    # Keep only the portfolio stocks present in returns, preserve order in portfolio
    pf = pf.set_index("Stock").loc[stocks].reset_index()
    stocks = pf["Stock"].tolist()
    n_stocks = len(stocks)

    # If input is prices, convert to log returns
    if is_price_input:
        df[stocks] = np.log(df[stocks]).diff()
        df = df.dropna(subset=stocks).reset_index(drop=True)

    # Coerce numeric and drop rows with NaN across used stocks
    df[stocks] = df[stocks].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=stocks).reset_index(drop=True)
    if len(df) == 0:
        raise ValueError("No valid rows after cleaning returns for portfolio stocks.")

    X = df[stocks].to_numpy(dtype=float)

    # Fit marginals per stock
    fitted = {}
    for s in stocks:
        dist_type = pf.loc[pf["Stock"] == s, "Distribution"].values[0]
        fitted[s] = fit_distribution(X[:, stocks.index(s)], dist_type)

    # Map historical returns to uniforms via each marginal CDF
    U = np.zeros_like(X)
    eps = np.finfo(float).eps
    for j, s in enumerate(stocks):
        U[:, j] = fitted[s].cdf(X[:, j])
    U = np.clip(U, eps, 1 - eps)

    # Probit transform to Z
    Z = stats.norm.ppf(U)

    # Spearman correlation (rank-based). More robust than Pearson on tails.
    # We'll use pairwise Spearman on columns of Z, then PSD-fix it.
    spearman_corr = np.eye(n_stocks)
    for i in range(n_stocks):
        for j in range(i + 1, n_stocks):
            corr, _ = spearmanr(Z[:, i], Z[:, j])
            if not np.isfinite(corr):
                corr = 0.0
            spearman_corr[i, j] = corr
            spearman_corr[j, i] = corr

    Sigma = nearest_psd(spearman_corr)
    # Cholesky
    L = np.linalg.cholesky(Sigma)

    # Simulate Z ~ N(0, Sigma)
    Z_sims = rng_local.standard_normal(size=(n_simulations, n_stocks)).dot(L.T)
    U_sims = stats.norm.cdf(Z_sims)

    # Map uniforms back to returns via each marginal PPF
    sim_returns = np.zeros_like(U_sims)
    for j, s in enumerate(stocks):
        u = np.clip(U_sims[:, j], eps, 1 - eps)
        sim_returns[:, j] = fitted[s].ppf(u)

    # Per-stock PnL distribution and portfolio aggregation
    results = []
    # Current values
    current_values = []
    for j, s in enumerate(stocks):
        holding = float(pf.loc[pf["Stock"] == s, "Holding"].values[0])
        start_price = float(pf.loc[pf["Stock"] == s, "Starting Price"].values[0])
        current_value = holding * start_price
        current_values.append(current_value)

    current_values = np.array(current_values)
    total_current_value = float(np.sum(current_values))

    # Per stock PnL
    per_stock_pnl = []
    for j, s in enumerate(stocks):
        holding = float(pf.loc[pf["Stock"] == s, "Holding"].values[0])
        start_price = float(pf.loc[pf["Stock"] == s, "Starting Price"].values[0])
        sim_prices = start_price * (1.0 + sim_returns[:, j])
        sim_values = holding * sim_prices
        pnl = sim_values - (holding * start_price)
        per_stock_pnl.append(pnl)

    per_stock_pnl = np.column_stack(per_stock_pnl)  # shape (n_sims, n_stocks)

    # Total portfolio PnL
    total_sim_values = np.sum([
        float(pf.loc[pf["Stock"] == s, "Holding"].values[0]) *
        (float(pf.loc[pf["Stock"] == s, "Starting Price"].values[0]) * (1.0 + sim_returns[:, j]))
        for j, s in enumerate(stocks)
    ], axis=0)
    total_pnl = total_sim_values - total_current_value

    # Build result rows for each alpha
    # For each stock
    rows = []
    for j, s in enumerate(stocks):
        row = {"Stock": s}
        for alpha in alphas:
            var_abs, es_abs = calculate_var_es_from_pnl(per_stock_pnl[:, j], alpha)
            var_pct = var_abs / current_values[j] if current_values[j] != 0 else np.nan
            es_pct = es_abs / current_values[j] if current_values[j] != 0 else np.nan
            level = int(round((1 - alpha) * 100))
            row[f"VaR{level}"] = var_abs
            row[f"ES{level}"] = es_abs
            row[f"VaR{level}_Pct"] = var_pct
            row[f"ES{level}_Pct"] = es_pct
        rows.append(row)

    # Total
    row_total = {"Stock": "Total"}
    for alpha in alphas:
        t_var_abs, t_es_abs = calculate_var_es_from_pnl(total_pnl, alpha)
        t_var_pct = t_var_abs / total_current_value if total_current_value != 0 else np.nan
        t_es_pct = t_es_abs / total_current_value if total_current_value != 0 else np.nan
        level = int(round((1 - alpha) * 100))
        row_total[f"VaR{level}"] = t_var_abs
        row_total[f"ES{level}"] = t_es_abs
        row_total[f"VaR{level}_Pct"] = t_var_pct
        row_total[f"ES{level}_Pct"] = t_es_pct
    rows.append(row_total)

    out = pd.DataFrame(rows)

    # Order as portfolio order + Total
    order = stocks + ["Total"]
    out["__ord"] = out["Stock"].apply(lambda s: order.index(s) if s in order else 999)
    out = out.sort_values("__ord").drop(columns="__ord").reset_index(drop=True)

    return out


def main():
    portfolio_df = pd.read_csv(PORTFOLIO_PATH)
    returns_df = pd.read_csv(RETURNS_PATH)

    results_df = calculate_var_es_copula(
        returns_df=returns_df,
        portfolio_df=portfolio_df,
        alphas=ALPHAS,
        n_simulations=N_SIMS,
        random_seed=RANDOM_SEED,
        is_price_input=IS_PRICE_INPUT
    )

    # Print CSV-style text (no file writing)
    print(results_df.to_csv(index=False, float_format="%.15f"), end="")

# Run this cell
main()

Stock,VaR95,ES95,VaR95_Pct,ES95_Pct,VaR99,ES99,VaR99_Pct,ES99_Pct
A,94.278627091835006,117.382376574125701,0.047139313545918,0.058691188287063,131.885412082153380,151.126644604280813,0.065942706041077,0.075563322302140
B,109.090426249169013,154.004163220760461,0.036363475416390,0.051334721073587,177.276061399615628,234.178229326971234,0.059092020466539,0.078059409775657
Total,153.107936491265718,200.851857180817575,0.030621587298253,0.040170371436164,227.238069839229865,282.105248559364838,0.045447613967846,0.056421049711873
